# TED 03 — Notebook 03: Análise de erros, robustez e aplicabilidade ao MVP
**Pergunta-guia:** onde o modelo erra, por quê, e em que condições ele pode ser usado no assistente da Inforcell?

Requer o notebook 02 executado (usa `models/classificador_falhas_treino2018.joblib`).

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import modelagem as M, graficos as g
g.aplicar_estilo()
ROOT = Path("../..").resolve()
RJ = ROOT / "reports" / "resultados_modelagem.json"
R = json.loads(RJ.read_text()) if RJ.exists() else {}
def salvar_R(): RJ.write_text(json.dumps(R, ensure_ascii=False, indent=1, default=float))
import joblib
from sklearn.base import clone
from sklearn.metrics import f1_score, log_loss
df = M.carregar_dados(); tr, te = M.dividir_temporal(df)
COLS = [M.COL_TEXTO, M.COL_APARELHO]
mdl = joblib.load(ROOT / "models" / "classificador_falhas_treino2018.joblib")
CL = mdl.classes_; s = mdl.predict_proba(te[COLS]); te["pred"] = CL[s.argmax(1)]; te["conf"] = s.max(1); te["acerto"] = te["pred"] == te[M.ALVO]
print("acurácia no teste:", round(te["acerto"].mean(), 3))

acurácia no teste: 0.716


## 1. Desempenho por segmento (idioma, aparelho, tamanho do texto)
Se o desempenho cai muito em algum segmento, o modelo não é confiável ali — e é nesse ponto que se decide quando o assistente deve “pedir mais informações” em vez de sugerir uma causa.

In [2]:
te["faixa_tam"] = te["n_palavras"].map(M.faixa_tamanho)
te["idioma_g"] = te["idioma"].map(lambda x: x if x in ("en", "nl", "de") else ("indefinido" if x == "indefinido" else "outros (fr, it, no, es)"))
def seg(col):
    r = []
    for k, d in te.groupby(col):
        r.append({"grupo": k, "n": int(len(d)), "acuracia": float(d["acerto"].mean()), "top3": float(M.topk_acuracia(d[M.ALVO].values, s[d.index], CL, 3)),
                  "f1_macro": float(f1_score(d[M.ALVO], d["pred"], average="macro", zero_division=0))})
    return pd.DataFrame(r)
S = {"idioma": seg("idioma_g"), "aparelho": seg("categoria"), "tamanho": seg("faixa_tam")}
R["segmentos"] = {k: v.to_dict("records") for k, v in S.items()}
fig, axs = plt.subplots(1, 3, figsize=(12, 3.9), sharey=True)
for ax, (nome, d), tit in zip(axs, S.items(), ["Idioma do texto", "Tipo de aparelho", "Tamanho do relato"]):
    d = d.copy(); ax.bar(d["grupo"], d["acuracia"] * 100, color=g.SERIES[0], width=.6, label="top-1")
    ax.scatter(d["grupo"], d["top3"] * 100, color=g.SERIES[1], zorder=3, s=36, label="top-3")
    for x, (a, n) in enumerate(zip(d["acuracia"], d["n"])): ax.text(x, a * 100 + 1.5, f"{a*100:.0f}%\nn={n}", ha="center", fontsize=7.5, color=g.TXT2)
    ax.set_title(tit); ax.grid(axis="x", visible=False); ax.tick_params(axis="x", labelrotation=25, labelsize=8); ax.set_ylim(0, 112)
axs[0].set_ylabel("% de acerto"); axs[0].legend(loc="lower left", fontsize=8)
fig.tight_layout(); g.salvar(fig, "fig18_desempenho_segmentos"); plt.show()
pd.concat(S, names=["segmento"]).round(3)

grupo     n  acuracia   top3  f1_macro
segmento                                                            
idioma   0                       de   261     0.571  0.843     0.526
         1                       en  1056     0.754  0.918     0.686
         2               indefinido   131     0.725  0.893     0.618
         3                       nl   297     0.717  0.869     0.588
         4  outros (fr, it, no, es)    50     0.640  0.860     0.474
aparelho 0                  Celular   453     0.733  0.909     0.579
         1                  Desktop   182     0.615  0.835     0.457
         2                 Notebook   938     0.707  0.888     0.649
         3                   Tablet   222     0.802  0.946     0.587
tamanho  0             1–3 palavras   702     0.765  0.916     0.672
         1             4–6 palavras   539     0.738  0.898     0.678
         2              7+ palavras   554     0.632  0.866     0.592

## 2. Erros: o que o modelo confunde e por quê

In [3]:
err = te[~te["acerto"]].copy()
alta = err.sort_values("conf", ascending=False)
alta = alta[(alta["n_palavras"] <= 12) & (alta["idioma"] == "en")].head(8)
R["exemplos_erros"] = [{"texto": r["problema"][:90], "verdadeiro": r[M.ALVO], "previsto": r["pred"], "confianca": float(r["conf"])} for _, r in alta.iterrows()]
print(f"{len(err)} erros em {len(te)} | erros com confiança >= 0.5: {(err['conf']>=.5).sum()}")
alta[["problema", M.ALVO, "pred", "conf"]].round(2)

510 erros em 1795 | erros com confiança >= 0.5: 157


,problema,categoria_falha,pred,conf
1695,Not booting,Energia / bateria,Inicialização (boot),0.99
71,Periodical screen shutdown,Software / sistema,Tela,0.99
1537,"New hard drive, additional memory",Desempenho / superaquecimento,Armazenamento,0.99
1212,Touchpad and keyboard faulty,Outros,Botões / teclado / touchpad,0.98
1523,computer tower needs hard drive wiping,Software / sistema,Armazenamento,0.92
38,Charging/power problem,Conectores e portas,Energia / bateria,0.92
923,Swollen/failed battery. Needs new battery and SSD,Outros,Energia / bateria,0.92
1617,Touch screen cables and buttons damaged,Placa / dano interno,Tela,0.89


**Como ler esta tabela:** vários “erros” de alta confiança são, na verdade, **rótulos discutíveis** (o texto é compatível com as duas categorias). Isso é coerente com o que a TED 02 já havia mostrado: a mesma frase aparece com rótulos diferentes e 26% dos casos são indeterminados até para voluntários experientes.

In [4]:
# (a) o teste mede memorização? textos idênticos ao treino vs. textos inéditos
vistos = set(tr[M.COL_TEXTO]); te["visto"] = te[M.COL_TEXTO].isin(vistos)
v = te.groupby("visto")["acerto"].agg(["mean", "size"]); print(v.round(3))
R["visto_vs_inedito"] = {("visto" if k else "inedito"): {"n": int(r["size"]), "acuracia": float(r["mean"])} for k, r in v.iterrows()}
# (b) erros concentrados nos pares de classes semanticamente próximas?
print(pd.DataFrame(R["confusoes"]).assign(pct_da_classe=lambda d: (d.pct_da_classe*100).round(1)))

        mean  size
visto             
False  0.709  1705
True   0.844    90
                      verdadeiro                       previsto   n  \
0             Software / sistema  Desempenho / superaquecimento  43   
1              Energia / bateria           Inicialização (boot)  22   
2             Software / sistema                         Outros  21   
3                         Outros             Software / sistema  18   
4              Energia / bateria            Conectores e portas  14   
5                         Outros  Desempenho / superaquecimento  12   
6           Placa / dano interno              Energia / bateria  11   
7  Desempenho / superaquecimento             Software / sistema  10   

   pct_da_classe  
0           10.5  
1            8.0  
2            5.1  
3           20.7  
4            5.1  
5           13.8  
6           12.4  
7            6.2  


## 3. Divisão temporal × divisão aleatória (efeito da deriva)
Se a divisão aleatória render números muito melhores, parte do desempenho “aparente” vem de misturar eventos do mesmo período (mesmos voluntários, mesmos modelos de rótulo, mesmo estilo de texto).

In [5]:
rows = []
for seed in (0, 1, 2):
    a, b = M.dividir_aleatorio(df, len(te) / len(df), seed); m = clone(mdl).fit(a[COLS], a[M.ALVO]); sb = m.predict_proba(b[COLS]); pb = m.classes_[sb.argmax(1)]
    rows.append(M.metricas(b[M.ALVO], pb, sb, m.classes_))
alea = pd.DataFrame(rows).mean(); alea_dp = pd.DataFrame(rows).std()
temp = R["final"]["metricas"]
R["temporal_vs_aleatorio"] = {k: {"temporal": temp[k], "aleatorio": float(alea[k]), "aleatorio_dp": float(alea_dp[k])} for k in ("acuracia", "f1_macro", "top3")}
pd.DataFrame(R["temporal_vs_aleatorio"]).T.round(3)

,temporal,aleatorio,aleatorio_dp
acuracia,0.716,0.729,0.009
f1_macro,0.654,0.657,0.003
top3,0.895,0.915,0.005


## 4. Curva de aprendizagem: faltam dados?
Treinamos com frações crescentes do treino (3 repetições cada) e medimos o F1-macro no teste. Se a curva ainda sobe, mais dados (por exemplo, as Ordens de Serviço reais da Inforcell) tendem a ajudar.

In [6]:
fr = [.1, .2, .4, .6, .8, 1.0]; out = []
for f in fr:
    for seed in range(3):
        sub = tr.sample(frac=f, random_state=seed); m = clone(mdl).fit(sub[COLS], sub[M.ALVO]); pp = m.predict(te[COLS])
        out.append({"fracao": f, "n": len(sub), "f1_macro": f1_score(te[M.ALVO], pp, average="macro", zero_division=0), "acuracia": float((pp == te[M.ALVO]).mean())})
cl = pd.DataFrame(out).groupby(["fracao", "n"]).agg(f1=("f1_macro", "mean"), f1_dp=("f1_macro", "std"), acc=("acuracia", "mean")).reset_index()
R["curva_aprendizagem"] = cl.to_dict("records")
fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(cl["n"], cl["f1"], marker="o", color=g.SERIES[0], label="F1-macro"); ax.fill_between(cl["n"], cl["f1"] - cl["f1_dp"], cl["f1"] + cl["f1_dp"], color=g.SERIES[0], alpha=.15)
ax.plot(cl["n"], cl["acc"], marker="s", color=g.SERIES[1], label="Acurácia")
ax.set_xlabel("nº de registros de treino"); ax.set_ylabel("desempenho no teste"); ax.set_ylim(0, .85); ax.set_title("Curva de aprendizagem"); ax.legend(loc="lower right"); fig.tight_layout(); g.salvar(fig, "fig19_curva_aprendizagem"); plt.show()
cl.round(3)

,fracao,n,f1,f1_dp,acc
0,0.1,222,0.394,0.013,0.526
1,0.2,445,0.479,0.010,0.589
2,0.4,890,0.549,0.009,0.636
3,0.6,1334,0.594,0.008,0.677
4,0.8,1779,0.625,0.004,0.700
5,1.0,2224,0.654,0.000,0.716


## 5. Confiança: quando confiar na sugestão?
O modelo devolve uma probabilidade. Se ela for **bem calibrada**, o assistente pode usá-la para decidir entre “sugerir a causa” e “fazer mais perguntas”. Avaliamos (a) a calibração e (b) a *classificação seletiva*: acurácia obtida quando só respondemos acima de um limiar de confiança, e a cobertura (fração de casos respondidos) que sobra.

In [7]:
te["bin"] = pd.qcut(te["conf"], 8, duplicates="drop"); cal = te.groupby("bin").agg(conf=("conf", "mean"), acc=("acerto", "mean"), n=("acerto", "size")).reset_index(drop=True)
ece = float((cal["n"] * (cal["conf"] - cal["acc"]).abs()).sum() / cal["n"].sum())
sel = M.curva_confianca(te[M.ALVO].values, te["pred"].values, te["conf"].values, cortes=np.arange(0, .86, .05))
R["calibracao"] = {"ece": ece, "bins": cal.to_dict("records")}; R["selecao_confianca"] = sel.dropna().to_dict("records")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4.2))
a1.plot([0, 1], [0, 1], "--", color=g.CINZA, label="calibração perfeita"); a1.plot(cal["conf"], cal["acc"], marker="o", color=g.SERIES[0], label="modelo")
a1.set_xlabel("confiança média prevista"); a1.set_ylabel("acurácia observada"); a1.set_title(f"Calibração (ECE = {ece:.3f})"); a1.legend(loc="upper left"); a1.set_xlim(0, 1); a1.set_ylim(0, 1)
d = sel.dropna(); a2.plot(d["cobertura"] * 100, d["acuracia"] * 100, marker="o", color=g.SERIES[0])
for _, r in d[d["corte"].round(2).isin([.2, .3, .4, .5, .6, .7, .8])].iterrows(): a2.annotate(f"≥{r['corte']:.2f}", (r["cobertura"] * 100, r["acuracia"] * 100), textcoords="offset points", xytext=(4, 6), fontsize=8, color=g.TXT2)
a2.set_xlabel("% dos casos em que o assistente responde (cobertura)"); a2.set_ylabel("acurácia nesses casos (%)"); a2.set_title("Acurácia × cobertura por limiar de confiança")
fig.tight_layout(); g.salvar(fig, "fig20_confianca"); plt.show()
sel.round(3)

,corte,cobertura,acuracia,n
0,0.00,1.000,0.716,1795
1,0.05,1.000,0.716,1795
2,0.10,1.000,0.716,1795
3,0.15,0.999,0.717,1793
4,0.20,0.972,0.730,1745
5,0.25,0.913,0.753,1638
6,0.30,0.848,0.783,1523
7,0.35,0.781,0.804,1402
8,0.40,0.726,0.818,1303
9,0.45,0.667,0.843,1197


In [8]:
# limiar sugerido: o menor corte em que a acurácia top-1 chega a 80% (com pelo menos 10% de cobertura)
ok = sel[(sel["acuracia"] >= .80) & (sel["cobertura"] >= .10)]
R["limiar_sugerido"] = None if ok.empty else {k: float(v) for k, v in ok.iloc[0].to_dict().items()}
print("Limiar sugerido:", R["limiar_sugerido"])

Limiar sugerido: {'corte': 0.35000000000000003, 'cobertura': 0.781058495821727, 'acuracia': 0.8038516405135521, 'n': 1402.0}


## 6. O caso “não liga” (resultado central da TED 02)
A TED 02 propôs ordenar os testes pelas probabilidades **a priori** das causas do sintoma “não liga”. Aqui medimos se o modelo — que enxerga o texto e o aparelho — ordena as causas melhor do que essa ordem fixa. Usa-se somente o teste (2019–2020); a ordem fixa é calculada no treino.

In [9]:
m_te = te["problema"].fillna("").str.contains(M.PADRAO_NAO_LIGA); m_tr = tr["problema"].fillna("").str.contains(M.PADRAO_NAO_LIGA)
ordem_fixa = list(tr.loc[m_tr, M.ALVO].value_counts().index)
sub = te[m_te]; ss = s[sub.index]
def hit_fixa(k): return float(np.mean([y in ordem_fixa[:k] for y in sub[M.ALVO]]))
res = {k: {"modelo": float(M.topk_acuracia(sub[M.ALVO].values, ss, CL, k)), "ordem_fixa": hit_fixa(k)} for k in (1, 2, 3)}
R["nao_liga"] = {"n_teste": int(len(sub)), "n_treino": int(m_tr.sum()), "ordem_fixa_treino": ordem_fixa[:5], "acerto": res, "dist_teste": sub[M.ALVO].value_counts(normalize=True).head(5).round(3).to_dict()}
fig, ax = plt.subplots(figsize=(6.6, 3.9)); x = np.arange(3)
ax.bar(x - .2, [res[k]["ordem_fixa"] * 100 for k in (1, 2, 3)], .38, color=g.SERIES[1], label="Ordem fixa (a priori da TED 02)"); ax.bar(x + .2, [res[k]["modelo"] * 100 for k in (1, 2, 3)], .38, color=g.SERIES[0], label="Modelo (texto + aparelho)")
for i, k in enumerate((1, 2, 3)):
    ax.text(i - .2, res[k]["ordem_fixa"] * 100 + 1.5, f"{res[k]['ordem_fixa']*100:.0f}%", ha="center", fontsize=8.5); ax.text(i + .2, res[k]["modelo"] * 100 + 1.5, f"{res[k]['modelo']*100:.0f}%", ha="center", fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(["1ª causa", "até a 2ª causa", "até a 3ª causa"]); ax.grid(axis="x", visible=False); ax.set_ylim(0, 110); ax.set_ylabel("% dos casos com a causa correta")
ax.set_title(f"Sintoma “não liga” no teste (n = {len(sub)})"); ax.legend(loc="lower right", fontsize=8.5); fig.tight_layout(); g.salvar(fig, "fig21_nao_liga"); plt.show()
print(R["nao_liga"])

{'n_teste': 142, 'n_treino': 218, 'ordem_fixa_treino': ['Energia / bateria', 'Inicialização (boot)', 'Placa / dano interno', 'Tela', 'Software / sistema'], 'acerto': {1: {'modelo': 0.6338028169014085, 'ordem_fixa': 0.4507042253521127}, 2: {'modelo': 0.8661971830985915, 'ordem_fixa': 0.7816901408450704}, 3: {'modelo': 0.9154929577464789, 'ordem_fixa': 0.8450704225352113}}, 'dist_teste': {'Energia / bateria': 0.451, 'Inicialização (boot)': 0.331, 'Placa / dano interno': 0.063, 'Armazenamento': 0.063, 'Tela': 0.042}}


## 7. O que o modelo aprendeu (interpretação)
Palavras de maior peso por categoria na regressão logística (apenas o bloco de palavras; o bloco de caracteres captura variações e erros de digitação). Serve para checar se o modelo aprendeu **sinais técnicos plausíveis** e não artefatos.

In [10]:
termos = M.principais_termos(mdl, CL, top=6, prefixo="w"); R["termos_modelo"] = termos
pd.DataFrame({"categoria": list(termos), "termos de maior peso": [", ".join(v) for v in termos.values()]})

,categoria,termos de maior peso
0,Armazenamento,"hd, hard, ssd, memory, data, photos"
1,Botões / teclado / touchpad,"button, keyboard, trackpad, clavier, key, switch"
2,Carcaça / dano externo,"hinge, verbogen, defekt, casing, start button,..."
3,Conectores e portas,"usb, sim, jack, port, socket, red"
4,Dano por líquido,"waterschade, water, niet in, wasserschaden, no..."
5,Desempenho / superaquecimento,"slow, fan, lenteur, traag, change hd, langzaam"
6,Energia / bateria,"battery, power, charger, accu, akku, dead"
7,Inicialização (boot),"boot, booting, bios, avvio, not booting, turns on"
8,Outros,"gereinigt, protector, connectivity, ed, need r..."
9,Placa / dano interno,"platine, card, das, cpu, ribbon, board"


## 8. Sondagem qualitativa com frases em português
A base não tem **nenhum** texto em português (TED 02, Seção 11.2). Para dimensionar o risco, testamos o modelo com frases curtas escritas pela equipe como as de um técnico/cliente brasileiro. **Não é uma métrica** (poucas frases, “resposta esperada” definida por nós): serve apenas para mostrar o comportamento fora do idioma de treino.

In [11]:
from inferencia import sugerir_causas
frases = [("tela quebrada", "Celular", "Tela"), ("não carrega, só funciona com o cabo torto", "Celular", "Conectores e portas"), ("bateria estufada", "Celular", "Energia / bateria"),
          ("caiu na água e não liga mais", "Celular", "Dano por líquido"), ("não liga e nenhuma luz acende", "Notebook", "Energia / bateria"), ("muito lento, trava toda hora", "Notebook", "Desempenho / superaquecimento"),
          ("teclado com várias teclas falhando", "Notebook", "Botões / teclado / touchpad"), ("liga mas não dá vídeo", "Desktop", "Tela")]
linhas = []
for t, ap, esp in frases:
    r = sugerir_causas(t, ap, k=3, modelo=mdl)
    top = [x["categoria_falha"] for x in r["sugestoes"]]
    linhas.append({"frase": t, "aparelho": ap, "esperada": esp, "top1": top[0], "prob_top1": r["sugestoes"][0]["probabilidade"], "esperada_no_top3": esp in top, "confianca_alta": r["confianca_alta"]})
sond = pd.DataFrame(linhas); R["sondagem_pt"] = sond.to_dict("records")
print("top-1 correto:", int((sond["top1"] == sond["esperada"]).sum()), "de", len(sond), "| esperada no top-3:", int(sond["esperada_no_top3"].sum()), "de", len(sond))
sond

top-1 correto: 1 de 8 | esperada no top-3: 2 de 8


,frase,aparelho,esperada,top1,prob_top1,esperada_no_top3,confianca_alta
0,tela quebrada,Celular,Tela,Outros,0.255,False,False
1,"não carrega, só funciona com o cabo torto",Celular,Conectores e portas,Áudio / câmera / mídia,0.329,False,False
2,bateria estufada,Celular,Energia / bateria,Outros,0.301,True,False
3,caiu na água e não liga mais,Celular,Dano por líquido,Outros,0.219,False,False
4,não liga e nenhuma luz acende,Notebook,Energia / bateria,Desempenho / superaquecimento,0.238,False,False
5,"muito lento, trava toda hora",Notebook,Desempenho / superaquecimento,Desempenho / superaquecimento,0.565,True,True
6,teclado com várias teclas falhando,Notebook,Botões / teclado / touchpad,Desempenho / superaquecimento,0.268,False,False
7,liga mas não dá vídeo,Desktop,Tela,Placa / dano interno,0.268,False,False


## 9. Exemplo de uso no MVP (`src/inferencia.py`)
O módulo devolve as 3 causas mais prováveis, as probabilidades e um indicador de confiança (limiar sugerido na seção 5).

In [12]:
sugerir_causas("won't turn on, no charging light", "Notebook", modelo=mdl)

{'sugestoes': [{'categoria_falha': 'Energia / bateria',
   'probabilidade': 0.915},
  {'categoria_falha': 'Desempenho / superaquecimento', 'probabilidade': 0.025},
  {'categoria_falha': 'Conectores e portas', 'probabilidade': 0.024}],
 'confianca_alta': True,
 'cobertura_vocabulario': 0.83,
 'acao_recomendada': 'Iniciar pelos testes da 1ª causa e seguir a ordem sugerida.'}

In [13]:
salvar_R(); print("resultados salvos em", RJ)

resultados salvos em reports/resultados_modelagem.json
